# Chapter 29 — Opening the Model: Interpretability

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements-optional.txt  # Colab only; skip locally

zsh:1: command not found: pip


zsh:1: command not found: pip


## Create the data

Run once. Every dataset in this book is generated by code you can read — nothing is downloaded, so nothing can rot behind a dead link. This is the printed block from Chapter 14 (`code/ch14/gen_hr.py` in the repository).

In [2]:
import numpy as np, pandas as pd

rng = np.random.default_rng(11)
n = 1470
tenure   = np.clip(rng.gamma(2.2, 3.0, n), 0.2, 40).round(1)
salary   = np.clip(rng.normal(65000, 18000, n), 25000, 160000).round(-2)
overtime = (rng.random(n) < 0.28).astype(int)
commute  = np.clip(rng.gamma(2.0, 6.0, n), 1, 60).round(0)
satis    = np.clip(rng.normal(3.3, 0.95, n), 1, 5).round(1)
promo    = np.clip(rng.gamma(1.6, 1.6, n), 0, 15).round(1)
dept     = rng.choice(["Sales", "R&D", "Support"], n,
                      p=[0.32, 0.45, 0.23])

# the relationship the model will have to rediscover
z = (-2.55 + 1.15*overtime - 0.135*tenure - 0.60*(satis - 3.3)
     + 0.024*commute + 0.095*promo - 0.000014*(salary - 65000)
     + np.where(dept == "Sales", 0.45,
                np.where(dept == "Support", 0.20, 0.0)))
left = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

hr = pd.DataFrame({"Department": dept, "YearsAtCompany": tenure,
    "MonthlyIncome": (salary/12).round(0),
    "OverTime": np.where(overtime == 1, "Yes", "No"),
    "CommuteMinutes": commute, "JobSatisfaction": satis,
    "YearsSincePromotion": promo, "Attrition": left})
hr.to_csv("hr.csv", index=False)
print(f"{len(hr):,} employees, "
      f"attrition rate {hr['Attrition'].mean():.1%}")

1,470 employees, attrition rate 12.2%


## The chapter code

### Block 1  (`c1.py`)

In [3]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

hr = pd.read_csv("hr.csv")
X = pd.get_dummies(hr.drop(columns="Attrition"), drop_first=True)
y = hr["Attrition"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25,
                                      random_state=0, stratify=y)

rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=8,
                            random_state=0).fit(Xtr, ytr)
print(f"columns: {list(X.columns)}")
print(f"test ROC AUC: {roc_auc_score(yte, rf.predict_proba(Xte)[:, 1]):.3f}")

columns: ['YearsAtCompany', 'MonthlyIncome', 'CommuteMinutes', 'JobSatisfaction', 'YearsSincePromotion', 'Department_Sales', 'Department_Support', 'OverTime_Yes']
test ROC AUC: 0.669


### Block 2  (`c2.py`)

In [4]:
imp = pd.Series(rf.feature_importances_,
                index=X.columns).sort_values(ascending=False)
print("impurity importance, straight from the forest")
for k, v in imp.items():
    print(f"  {k:<22} {v:.3f}")

impurity importance, straight from the forest
  YearsAtCompany         0.193
  MonthlyIncome          0.192
  JobSatisfaction        0.184
  YearsSincePromotion    0.153
  CommuteMinutes         0.133
  OverTime_Yes           0.098
  Department_Sales       0.038
  Department_Support     0.010


### Block 3  (`c3.py`)

In [5]:
from sklearn.inspection import permutation_importance

r = permutation_importance(rf, Xte, yte, n_repeats=30,
                           random_state=0, scoring="roc_auc")
perm = pd.Series(r.importances_mean,
                 index=X.columns).sort_values(ascending=False)

print("permutation importance, measured on held-out data")
for k in perm.index:
    sd = r.importances_std[list(X.columns).index(k)]
    print(f"  {k:<22} {perm[k]:>+.4f}  (sd {sd:.4f})")

permutation importance, measured on held-out data
  YearsAtCompany         +0.0622  (sd 0.0245)
  OverTime_Yes           +0.0557  (sd 0.0169)
  Department_Sales       +0.0282  (sd 0.0118)
  JobSatisfaction        +0.0231  (sd 0.0215)
  CommuteMinutes         +0.0091  (sd 0.0107)
  YearsSincePromotion    +0.0070  (sd 0.0152)
  Department_Support     -0.0046  (sd 0.0030)
  MonthlyIncome          -0.0150  (sd 0.0208)


### Block 4  (`c4.py`)

In [6]:
from sklearn.inspection import partial_dependence

for feat in ["YearsAtCompany", "JobSatisfaction"]:
    pd_ = partial_dependence(rf, Xte, [list(X.columns).index(feat)],
                             kind="average", grid_resolution=6)
    grid = pd_["grid_values"][0]
    vals = pd_["average"][0]
    print(f"{feat}")
    for g, v in zip(grid, vals):
        print(f"   {g:>6.1f}  ->  predicted attrition {v:.3f}")

YearsAtCompany
      1.4  ->  predicted attrition 0.173
      4.3  ->  predicted attrition 0.109
      7.3  ->  predicted attrition 0.087
     10.2  ->  predicted attrition 0.075
     13.2  ->  predicted attrition 0.066
     16.1  ->  predicted attrition 0.070


JobSatisfaction
      1.8  ->  predicted attrition 0.193
      2.4  ->  predicted attrition 0.147
      3.0  ->  predicted attrition 0.118
      3.5  ->  predicted attrition 0.080
      4.1  ->  predicted attrition 0.081
      4.7  ->  predicted attrition 0.077


### Block 5  (`c5.py`)

In [7]:
import shap

expl = shap.TreeExplainer(rf)
sv = expl.shap_values(Xte, check_additivity=False)[:, :, 1]   # class 1

mean_abs = pd.Series(np.abs(sv).mean(0),
                     index=X.columns).sort_values(ascending=False)
print("mean |SHAP| — average size of each feature's contribution")
for k, v in mean_abs.items():
    print(f"  {k:<22} {v:.4f}")

<site-packages>/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mean |SHAP| — average size of each feature's contribution
  JobSatisfaction        0.0328
  OverTime_Yes           0.0311
  YearsAtCompany         0.0311
  MonthlyIncome          0.0257
  CommuteMinutes         0.0152
  YearsSincePromotion    0.0138
  Department_Sales       0.0124
  Department_Support     0.0014


### Block 6  (`c6.py`)

In [8]:
import shap
expl = shap.TreeExplainer(rf)
sv = expl.shap_values(Xte, check_additivity=False)[:, :, 1]

i = int(np.argmax(rf.predict_proba(Xte)[:, 1]))   # highest-risk person
base = expl.expected_value[1]
row = Xte.iloc[i]

print(f"employee {Xte.index[i]}   predicted risk "
      f"{rf.predict_proba(Xte)[i, 1]:.3f}   baseline {base:.3f}")
order = np.argsort(-np.abs(sv[i]))
for j in order:
    print(f"  {X.columns[j]:<22} = {row.iloc[j]:>8}   "
          f"pushes {sv[i, j]:>+.4f}")
print(f"  {'sum of pushes':<22}   {sv[i].sum():>+.4f}"
      f"   -> {base + sv[i].sum():.3f}")

employee 154   predicted risk 0.397   baseline 0.122
  OverTime_Yes           =        1   pushes +0.0846
  JobSatisfaction        =      1.7   pushes +0.0730
  YearsAtCompany         =      1.8   pushes +0.0634
  MonthlyIncome          =   4575.0   pushes +0.0355
  YearsSincePromotion    =      4.2   pushes +0.0217
  Department_Sales       =        0   pushes -0.0204
  CommuteMinutes         =     30.0   pushes +0.0152
  Department_Support     =        0   pushes +0.0026
  sum of pushes            +0.2755   -> 0.397
